# small-vs-frontier: Colab GPU steps

Every GPU step of the project, in order. Claude Code drives this notebook through colab-mcp; the same cells can be run by hand. Runtime: **T4 GPU** (Runtime → Change runtime type).

## Phase 1: setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv || echo "NO GPU"

Mounting Drive needs you to approve a Google popup, so run this cell by hand.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/small-vs-frontier")
for sub in ["checkpoints", "adapters", "results", "gguf"]:
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)
print(sorted(p.name for p in DRIVE_ROOT.iterdir()))

Add `HF_TOKEN` in the Secrets panel (🔑, notebook access on). Secrets can only be read when the cell is run **from the Colab UI**, so run the next cell by hand after every runtime restart. Processes started from the notebook inherit it.

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # never printed
print("HF_TOKEN set:", bool(os.environ["HF_TOKEN"]))

Pinned to the versions verified on a T4 (see `requirements-colab.txt`).

In [ ]:
%%capture pip_log
!pip install -q unsloth==2026.10.5 transformers==5.17.0 trl==1.15.0 peft==0.21.1 bitsandbytes==0.50.2 accelerate==1.15.0 datasets==4.8.5 pyyaml scikit-learn python-dotenv

In [ ]:
print("\n".join(pip_log.stdout.splitlines()[-15:]))
print(pip_log.stderr[-2000:])

Import check in a fresh process, so modules already loaded in the kernel can't hide a version clash.

In [ ]:
%%bash
python - <<'PYEOF' 2>&1 | grep -v -i "warn" | tail -25
import platform
import unsloth  # must be imported before transformers/trl/peft
import torch, transformers, trl, peft, bitsandbytes, datasets, accelerate
print("python", platform.python_version())
for m in (unsloth, torch, transformers, trl, peft, bitsandbytes, datasets, accelerate):
    print(f"{m.__name__}=={m.__version__}")
print("cuda available:", torch.cuda.is_available(), "| cuda", torch.version.cuda, "|", torch.cuda.get_device_name(0))
x = torch.randn(1024, 1024, device="cuda", dtype=torch.float16)
print("fp16 matmul ok:", (x @ x).isfinite().all().item())
PYEOF

## Phase 2: code and data

Project code lives on Drive (`small-vs-frontier/code`) so it survives runtime resets. Until the repo is pushed, Claude Code copies `src/`, `scripts/` and `configs/` there and checks each file's sha256 against the laptop copy.

In [ ]:
# Project code lives on Drive so it survives runtime resets.
# Once the repo is on GitHub this becomes: !git clone https://github.com/Ham12-3/fine-tune {CODE}
# Until then Claude Code copies src/, scripts/ and configs/ here and checks each file's sha256 against the laptop.
import hashlib
CODE = DRIVE_ROOT / "code"
for p in sorted(CODE.rglob("*")):
    if p.is_file() and "data" not in p.relative_to(CODE).parts[:1]:
        print(hashlib.sha256(p.read_bytes()).hexdigest()[:16], p.relative_to(CODE))

Rebuild the splits (downloads the pinned CSVs and checks their sha256). The hashes must match `data/processed/manifest.json` on the laptop.

In [ ]:
!cd {CODE} && PYTHONPATH=src python scripts/prepare_data.py
import json
m = json.loads((CODE / "data/processed/manifest.json").read_text())
print(m["labels_sha256"][:16], {k: v["sha256"][:16] for k, v in m["files"].items()})

## Phase 3: probe the candidate base models

Each probe loads the model in 4-bit, adds LoRA, times training steps and zero-shot generation, and writes `results/phase3/<model>.json`. They run one after another in the background.

In [ ]:
PROBE = DRIVE_ROOT / "results/phase3"
PROBE.mkdir(parents=True, exist_ok=True)
models = ["unsloth/Qwen3.5-2B", "unsloth/gemma-4-E2B-it", "unsloth/Phi-4-mini-instruct"]
cmds = " ; ".join(f"PYTHONPATH=src python scripts/probe_model.py {m} --out {PROBE}/{m.split('/')[1]}.json "
                  f"> {PROBE}/{m.split('/')[1]}.log 2>&1" for m in models)
!cd {CODE} && nohup bash -c "{cmds}" > /dev/null 2>&1 &

In [ ]:
import re
for log in sorted(PROBE.glob("*.log")):
    lines = [re.sub(r"\x1b\[[0-9;]*m", "", l) for l in log.read_text(errors="replace").splitlines()]
    lines = [l for l in lines if l.strip() and "%|" not in l and "Warning" not in l]
    print(f"== {log.name} ({len(lines)} lines)\n" + "\n".join(lines[-12:]))
!nvidia-smi --query-gpu=memory.used,utilization.gpu --format=csv,noheader

In [ ]:
import json
for f in sorted(PROBE.glob("*.json")):
    r = json.loads(f.read_text())
    r.pop("prompt_tail")
    print(json.dumps(r))

## Phase 4: small-model zero-shot baseline

Same system prompt and parser as Claude zero-shot; greedy, thinking off, 4-bit base model. Started with `subprocess.Popen(..., start_new_session=True)` so it runs detached from the kernel (a shell `&` launch once hung the cell when the browser connection dropped). Note: on a T4 (no bfloat16) Unsloth computes Qwen3.5 in float32; fp16 overflows on this architecture.

In [ ]:
import subprocess, os
LOGS = DRIVE_ROOT / "logs"
LOGS.mkdir(exist_ok=True)
proc = subprocess.Popen(["python", "scripts/run_small_zeroshot.py"], cwd=CODE, env={**os.environ, "PYTHONPATH": "src"},
                        stdout=open(LOGS / "small_zeroshot.log", "w"), stderr=subprocess.STDOUT, start_new_session=True)
print("started pid", proc.pid)

In [ ]:
# progress (waits up to 90 s for new output)
import re, time
def show(wait=90):
    log = LOGS / "small_zeroshot.log"
    n0 = len(log.read_text(errors="replace"))
    for _ in range(wait // 5):
        if len(log.read_text(errors="replace")) != n0 or proc.poll() is not None:
            break
        time.sleep(5)
    lines = [re.sub(r"\x1b\[[0-9;]*m", "", l) for l in log.read_text(errors="replace").splitlines()]
    print("\n".join(l for l in lines if l.strip() and "%|" not in l and "Warning" not in l)[-2000:])
    print("exit code:", proc.poll())
show()

## Phase 5: QLoRA sweep

`scripts/train_sweep.py` trains one adapter per training fraction (nested subsets from Phase 2), checkpoints to `checkpoints/f<frac>` on Drive every 100 steps, resumes from the latest checkpoint, saves the adapter to `adapters/f<frac>`, scores it on the validation split and writes `done.json` (finished runs are skipped). Measured on the T4: ~1.3 s per step (batch 16, ~44 tokens per example, float32 compute).

Dry run first: prints one encoded example and confirms the loss covers only the label tokens.

In [ ]:
import subprocess, os, re, time, shutil, signal, json
r = subprocess.run(["python", "scripts/train_sweep.py", "--dry-run"], cwd=CODE, env={**os.environ, "PYTHONPATH": "src"},
                   capture_output=True, text=True, timeout=900)
print("\n".join(l for l in r.stdout.splitlines() if l.startswith(("prompt", "loss on", "tokens/"))))

Resume test: a throwaway 150-step run is killed after `checkpoint-100`, then restarted with the same command; it must log `resuming from .../checkpoint-100` and record 2 sessions.

In [ ]:
def launch(args, logname):
    return subprocess.Popen(["python", "-u", "scripts/train_sweep.py", *args], cwd=CODE, env={**os.environ, "PYTHONPATH": "src"},
                            stdout=open(f"/content/{logname}", "a"), stderr=subprocess.STDOUT, start_new_session=True)
RT = ["--fractions", "0.10", "--tag", "resume_test", "--max-steps", "150"]
ck = DRIVE_ROOT / "checkpoints/f0.10_resume_test"
proc = launch(RT, "resume_test.log")
while not (ck / "checkpoint-100" / "trainer_state.json").exists() and proc.poll() is None:
    time.sleep(5)
time.sleep(20)
os.killpg(proc.pid, signal.SIGKILL)
proc = launch(RT, "resume_test.log")
proc.wait()
print([l for l in open("/content/resume_test.log").read().splitlines() if re.search(r"resuming|starting from|val acc", l)])
print(json.loads((ck / "sessions.json").read_text()))

The sweep itself (all fractions; finished ones are skipped). The monitor cell keeps the notebook busy so Colab doesn't idle out, and copies the log to Drive.

In [ ]:
proc = launch([], "sweep.log")
print("started pid", proc.pid)

In [ ]:
def monitor(minutes=20, tail=8):
    t0 = time.time()
    while proc.poll() is None and time.time() - t0 < minutes * 60:
        time.sleep(15)
    shutil.copy("/content/sweep.log", LOGS / "sweep.log")
    lines = [l for l in open("/content/sweep.log", errors="replace").read().splitlines()
             if re.search(r"^== |'loss'|val acc|resuming|starting from|Error|Traceback", l)]
    print("\n".join(lines[-tail:]))
    print("exit:", proc.poll(), "| done:", sorted(p.parent.name for p in (DRIVE_ROOT / "adapters").glob("f*/done.json")))
monitor(20)